# 재활용 폐기물 실시간 감지 — 향후 과제 및 개선 방향

## 1. 개요

### 1.1 현재 실험 성과 요약

YOLO26s 모델을 활용하여 5종 폐기물(고철류·비닐·유리병·캔류·형광등) 실시간 감지 모델을 학습·평가하였다.

| 실험 | 모델 | Epochs | mAP@50 | mAP@50-95 | 비고 |
|------|------|--------|--------|-----------|------|
| baseline | YOLO26s | 50 | **0.9624** | **0.9164** | 최종 제출 모델 |

- **데이터**: AI Hub 「생활 폐기물 이미지」 (dataSetSn=71385) 기반
- **입력 해상도 필터**: 1920×1080 (16:9) 이미지만 사용
- **세션 필터**: 건당 유효 이미지 ≥ 5장인 세션만 학습에 활용
- **클래스당 2,000장, 총 10,000장** (PIL 무결성 검사 통과)
- **학습 환경**: WSL2 + RTX 4060 Ti 8GB, CUDA 12.1, PyTorch 2.5.1

### 1.2 핵심 한계점

| 한계 | 설명 | 영향 |
|------|------|------|
| **단일 데이터셋 편향** | AI Hub 데이터 1종만 사용 — 배경·조명·촬영 거리의 다양성 부족 | 실환경 일반화 성능 저하 |
| **다중 클래스 공존 부재** | 대부분의 이미지에 1종 폐기물만 등장 | 실제 분리수거 환경과 괴리 |
| **조명·각도 편중** | 정면·주간 이미지 비율이 높음 | 야간·역광·기울어진 폐기물 오탐 |
| **문제 정의 미정립** | 탐지 vs. 분류 vs. 이상 감지 중 어느 것을 목표로 하는지 불명확 | 평가 지표 선택 흔들림 |
| **신뢰도 임계값 미최적화** | 클래스 수에 따라 최적 임계값이 달라지나 고정값 사용 | 오탐/미탐 비율 불균형 |

### 1.3 참조 데이터셋 (AI Hub)

| # | 데이터셋 | AI Hub ID | 특징 |
|---|---------|-----------|------|
| 1 | 생활 폐기물 이미지 | dataSetSn=71385 | 본 실험에서 사용한 주 데이터셋 |
| 2 | 재활용 가능 폐기물 이미지 | dataSetSn=495 | 클래스 구분 체계가 다름 — 혼합 전략 검토 필요 |
| 3 | 도심지 생활 폐기물 감지 영상 | dataSetSn=71362 | 실외 CCTV 영상 기반 — 실환경 적용성 높음 |

---

## 2. 향후 과제 전체 구성도

```
┌─────────────────────────────────────────────────────────────────┐
│              재활용 폐기물 감지 — 향후 개선 로드맵               │
├────────────────┬────────────────┬────────────────┬──────────────┤
│  과제 A        │  과제 B        │  과제 C        │  과제 D      │
│  데이터셋 확장 │  다중 클래스   │  조명·각도     │  문제 정의   │
│  (다중 출처    │  공존 데이터   │  다양성 증강   │  고도화      │
│   혼합 전략)   │  구성          │                │              │
├────────────────┴────────────────┴────────────────┴──────────────┤
│  과제 E: 신뢰도 점수 분석 & 클래스 수별 임계값 최적화            │
├─────────────────────────────────────────────────────────────────┤
│  과제 F: 2단계 탐지 아키텍처 (분리수거통 이상 감지 서비스)       │
└─────────────────────────────────────────────────────────────────┘
```

각 과제는 독립적으로 진행 가능하며, 과제 A~D가 완료된 후 E·F로 통합하는 방식을 권장한다.

---

## 3. 과제 A — 다중 데이터셋 혼합 전략

### 3.1 배경

현재 실험에서는 AI Hub `dataSetSn=71385` 단일 출처만 사용하였다.  
단일 출처 데이터는 **촬영 환경(배경·조도·카메라 기종)이 균일**하여 모델이 폐기물 자체의 특징보다 배경 패턴에 과적합할 위험이 있다.

AI Hub에는 동일 도메인의 데이터셋이 복수 존재한다:

| 데이터셋 | ID | 촬영 환경 | 클래스 체계 |
|---------|----|----------|------------|
| 생활 폐기물 이미지 | 71385 | 실내 스튜디오 중심 | 고철류·비닐·유리병·캔류·형광등 (5종) |
| 재활용 가능 폐기물 이미지 | 495 | 실내+실외 혼합 | 종이·플라스틱·유리·금속·스티로폼 (6종) |
| 도심지 생활 폐기물 감지 영상 | 71362 | 실외 CCTV 영상 | 투기 폐기물 중심 (분류 방식 상이) |

### 3.2 클래스 매핑 전략

세 데이터셋의 클래스 체계가 다르므로, **공통 상위 클래스(super class)** 로 통합하거나 **클래스별 출처를 선별**하는 방식이 필요하다.

**방안 1 — 클래스별 최적 출처 선택 (권장)**

| 클래스 | 주 출처 | 보조 출처 | 이유 |
|--------|--------|----------|------|
| 캔류 | 71385 | 495(금속) | 형상 다양성 보완 |
| 유리병 | 71385 | 495(유리) | 투명도·색상 다양성 |
| 비닐 | 71385 | 71362 | 실외 투기 비닐 패턴 추가 |
| 고철류 | 71385 | 71362 | 야외 고철 이미지 보강 |
| 형광등 | 71385 | — | 유사 클래스 없음 |

**방안 2 — 출처 균형 샘플링**

```python
# 예시: 클래스당 2,000장 중 출처별 비율 설정
SOURCE_RATIO = {
    71385: 0.60,   # 기존 주 데이터셋 60%
    495:   0.25,   # 재활용 가능 폐기물 25%
    71362: 0.15,   # 도심지 영상 15%
}
```

### 3.3 혼합 시 주의사항

- **클래스 정의 충돌**: `495`의 「플라스틱」과 `71385`의 「비닐」은 재질이 다름 — 혼합 전 어노테이션 재검토 필수  
- **해상도 통일**: 각 출처의 이미지 해상도가 다를 수 있으므로 resize 전처리 파이프라인 통일  
- **세션 단위 분리 유지**: 데이터 누수 방지를 위해 출처별로도 세션 기반 train/valid/test 분리 적용  
- **출처 태그 보존**: 학습 후 출처별 성능 분석을 위해 파일명에 출처 ID 접두어 유지  

### 3.4 기대 효과

```
현재 (단일 출처)
  mAP@50 = 0.9624  ← 스튜디오 환경에서 높은 성능
  실환경 테스트 = ? ← 실외·야간·다양한 배경에서 성능 미확인

혼합 후 (예상)
  스튜디오 mAP@50 = 소폭 하락 가능 (0.93~0.96 추정)
  실환경 mAP@50   = 유의미한 향상 기대
  일반화 지수(G-mAP) = 스튜디오 + 실환경 평균 → 전반적 향상
```

---

## 4. 과제 B — 다중 클래스 공존 데이터 구성

### 4.1 현재 데이터의 구조적 문제

현재 학습 데이터의 대부분은 **이미지 1장에 1종의 폐기물**만 등장한다.  
EDA 단계에서 분석한 co-occurrence matrix에서도 클래스 간 동시 출현 빈도가 매우 낮음을 확인하였다.

```
이상적인 실환경 (분리수거통 앞)
┌─────────────────────────────┐
│  [캔]   [유리병]  [캔]      │
│     [비닐봉지]              │
│  [캔]       [고철]          │
└─────────────────────────────┘
  → 1장의 이미지에 3~5종이 동시 존재

현재 학습 데이터
┌─────────────────────────────┐
│                             │
│         [캔 1개]            │
│                             │
└─────────────────────────────┘
  → 1장에 1종, 단독 촬영
```

이로 인해 **실제 분리수거 환경에서 여러 폐기물이 섞여 있을 때 각 객체를 정확히 탐지하는 능력이 검증되지 않은 상태**다.

### 4.2 다중 클래스 공존 데이터 구성 방법

**방법 1 — Mosaic 증강 (즉시 적용 가능)**

Ultralytics의 내장 mosaic 증강을 활성화하면 4개의 이미지를 합쳐 다중 클래스 공존 효과를 낼 수 있다.

```python
model.train(
    data='data/dataset.yaml',
    mosaic=1.0,        # 기본값 — 100% 확률로 mosaic 적용
    mixup=0.1,         # 추가 mixup 증강
    copy_paste=0.1,    # Copy-Paste 증강 (객체 레벨)
)
```

**방법 2 — Copy-Paste 합성 (고품질, 수작업)**

클래스별 개별 폐기물 이미지에서 객체를 잘라내어 배경 이미지에 합성:

```python
# 개념 코드
def create_multi_class_image(bg_image, object_crops):
    """
    배경 이미지에 2~5종 폐기물 객체를 랜덤 배치하여 합성.
    - Poisson blending으로 경계 자연스럽게 처리
    - 겹침(IoU > 0.3)이 발생하면 재배치
    - 새로운 YOLO 어노테이션 자동 생성
    """
    ...
```

**방법 3 — 실제 혼합 촬영 데이터 추가 수집**

가장 현실적이지만 비용이 높음. AI Hub `71362`(도심지 폐기물 영상)에 혼합 배치 장면이 포함되어 있을 가능성이 있으므로 EDA 우선 수행.

### 4.3 다중 클래스 공존 데이터 목표 구성

| 구성 | 비율 | 내용 |
|------|------|------|
| 단일 클래스 이미지 | 40% | 현재와 동일 — 개별 객체 정밀 탐지 유지 |
| 2~3종 공존 이미지 | 40% | Mosaic + Copy-Paste 합성 |
| 4~5종 공존 이미지 | 20% | 실제 분리수거 환경 시뮬레이션 |

### 4.4 평가 지표 추가

다중 클래스 공존 성능을 별도로 측정하기 위해 평가 셋 분리:

```
test_single/   — 1종만 있는 이미지
test_multi/    — 2종 이상이 있는 이미지

비교 지표:
  mAP@50 (single) vs mAP@50 (multi)
  → 두 값의 차이가 클수록 다중 공존 일반화 부족
```

---

## 5. 과제 C — 조명·각도 다양성 증강

### 5.1 조명 다양성

현재 데이터셋은 **정상 실내 조명(주간 스튜디오 촬영)** 편향이 강하다.  
실제 분리수거 환경에서는 다음 조건이 빈번하다:

| 조명 조건 | 실환경 시나리오 | 현재 데이터 포함 여부 |
|----------|----------------|---------------------|
| 저조도(야간) | 야간 분리수거 투기 감시 | 매우 드묾 |
| 강조도(직사광선) | 옥외 수거 거점, 햇빛 반사 | 드묾 |
| 역광 | 창가·출입구 앞 | 드묾 |
| 형광등 깜빡임 | 실내 수거장 | 거의 없음 |

**Albumentations 기반 조명 증강 적용 예시:**

```python
import albumentations as A

lighting_aug = A.Compose([
    A.OneOf([
        A.RandomBrightnessContrast(brightness_limit=(-0.4, -0.1)),  # 저조도
        A.RandomBrightnessContrast(brightness_limit=(0.1, 0.4)),    # 고조도
        A.CLAHE(clip_limit=4.0),                                     # 대비 강화
        A.RandomShadow(num_shadows_upper=2),                         # 그림자
        A.RandomFog(fog_coef_upper=0.3),                             # 안개(실외)
    ], p=0.5),
    A.ColorJitter(brightness=0.2, contrast=0.2,
                  saturation=0.2, hue=0.1, p=0.3),
], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels']))
```

### 5.2 각도·방향 다양성

폐기물은 투기 또는 수거 과정에서 **다양한 방향으로 놓이거나 쌓인다.**  
현재 데이터셋은 정면·직립 자세가 대부분이다.

| 각도 조건 | 예시 | 탐지 난이도 |
|----------|------|------------|
| 직립 (정면) | 세워진 캔, 직립 유리병 | 낮음 |
| 눕혀진 상태 | 옆으로 누운 캔·유리병 | 중간 |
| 뒤집힌 상태 | 거꾸로 놓인 형광등 | 높음 |
| 기울어진 상태 | 45° 기울어진 병 | 중간 |
| 겹쳐진 상태 | 캔 위에 또 다른 캔 | 높음 |

**Ultralytics 내장 회전·뒤집기 증강 활성화:**

```python
model.train(
    degrees=30,       # ±30° 회전
    fliplr=0.5,       # 좌우 반전 50%
    flipud=0.1,       # 상하 반전 10% (폐기물은 상하도 의미 있음)
    perspective=0.001,# 원근 변환
    shear=5.0,        # 전단 변환
)
```

### 5.3 형상 다양성 확보 전략

같은 클래스라도 형상이 다양하다:

```
캔류:
  ├─ 원통형 음료캔 (330ml, 500ml)
  ├─ 납작하게 찌그러진 캔
  ├─ 개봉 전 / 개봉 후
  └─ 음료캔 / 통조림캔

비닐:
  ├─ 봉지 형태 (부풀어 오른 상태)
  ├─ 납작하게 접힌 상태
  ├─ 뭉쳐진 상태
  └─ 단일 비닐 / 여러 장 겹침

형광등:
  ├─ 직관형 (T8, T5)
  ├─ 원형 (환형)
  └─ 콤팩트형 (U자, 나선형)
```

데이터 수집 시 **형상 체크리스트**를 만들어 각 변형 형태가 최소 N장씩 포함되도록 관리하는 것이 중요하다.

---

## 6. 과제 D — 문제 정의 고도화

### 6.1 문제 정의가 모델 평가에 미치는 영향

"재활용 폐기물 감지"라는 도메인 아래에도 **어떤 문제를 풀 것인지에 따라 적합한 모델 구조·평가 지표·배포 형태가 완전히 달라진다.**

| 문제 정의 | 핵심 질문 | 적합한 모델 | 주요 평가 지표 |
|----------|----------|------------|---------------|
| **A. 단순 분류** | 이 이미지에 어떤 폐기물이 있나? | ResNet / ViT (분류기) | Top-1 Acc, F1 |
| **B. 객체 탐지** | 어디에 어떤 폐기물이 있나? | YOLO26s (현재) | mAP@50, mAP@50-95 |
| **C. 이상 감지** | 분리수거통에 틀린 폐기물이 섞였나? | YOLO26 + 규칙 엔진 | Recall(이상), FPR |
| **D. 투기 감시** | 지정 장소 외에 폐기물을 버렸나? | 탐지 + 추적(ByteTrack) | 탐지율, 지연 시간 |

### 6.2 실사용 시나리오 분석

가장 현실적이고 높은 가치를 갖는 시나리오는 **"분리수거통의 이상 감지"** 이다:

```
실환경 시나리오
─────────────────────────────────────────────
 [캔류 전용 수거통]
  ├─ 정상: 캔만 들어 있음 → 이상 없음
  ├─ 이상: 유리병이 섞여 있음 → 경고 발생
  └─ 이상: 비닐이 투기됨 → 경고 발생

이 경우 모델에 필요한 것:
  1. 통 내부 폐기물의 종류를 탐지
  2. 해당 수거통의 허용 클래스 목록과 비교
  3. 허용 외 클래스 감지 시 알림 발송
```

### 6.3 문제 정의에 따른 평가 지표 재설계

이상 감지 문제로 정의할 경우, **Recall(재현율)이 Precision보다 중요하다:**

```
오탐(FP) 비용: 정상 수거통에 경고 → 불편함 (낮은 비용)
미탐(FN) 비용: 이상 폐기물 미감지 → 분리수거 실패 (높은 비용)

→ Recall 최대화 (≥ 0.95) 목표
→ Precision은 0.80 이상으로 허용
→ 현재 mAP@50-95 최적화 기준은 부적절할 수 있음
```

**문제 정의별 권장 평가 체계:**

| 문제 정의 | Recall 목표 | Precision 목표 | 임계값 조정 방향 |
|----------|------------|----------------|----------------|
| 이상 감지 | ≥ 0.95 | ≥ 0.80 | 낮게 (0.3~0.4) |
| 투기 감시 | ≥ 0.90 | ≥ 0.85 | 낮게 (0.35~0.45) |
| 수거 최적화 | ≥ 0.85 | ≥ 0.90 | 높게 (0.5~0.6) |

### 6.4 실제 배포 환경 정의

문제 정의를 확정하기 전에 다음을 명확히 해야 한다:

```
체크리스트:
  ☐ 카메라 위치: 수거통 상단 고정? 외부 CCTV?
  ☐ 촬영 대상: 수거통 내부? 투기 행위?
  ☐ 운영 주체: 아파트 관리소? 지자체? 기업?
  ☐ 대응 방식: 실시간 알림? 일별 리포트?
  ☐ 처리 위치: 엣지(카메라 내장)? 서버 기반?
```

---

## 7. 과제 E — 신뢰도 점수 분석 및 클래스 수별 임계값 최적화

### 7.1 클래스 수가 신뢰도 분포에 미치는 영향

YOLO 계열 모델은 **클래스 수가 많을수록 소프트맥스 경쟁이 강해져** 개별 클래스의 최대 신뢰도가 낮아지는 경향이 있다.

```
동일한 캔 이미지, 클래스 수별 신뢰도 변화 (가상 예시)
────────────────────────────────────────────
 2클래스 (캔 vs 비캔):    confidence ≈ 0.95
 3클래스 (캔·유리·비닐):  confidence ≈ 0.88
 4클래스 (+ 고철):        confidence ≈ 0.82
 5클래스 (+ 형광등):      confidence ≈ 0.77

→ 클래스 수가 늘수록 최적 임계값도 낮아져야 함
→ 고정 임계값(default=0.25)은 5클래스에서 오탐 증가
```

### 7.2 클래스 수별 실험 설계

| 실험 | 클래스 구성 | 활용 시나리오 |
|------|-----------|-------------|
| Exp-2cls | 캔류 vs 비캔류 (나머지 통합) | 캔 전용 수거통 감시 |
| Exp-3cls | 캔류·유리병·기타 | 3-분리 수거통 시스템 |
| Exp-4cls | 캔류·유리병·비닐·기타 | 4-분리 수거통 시스템 |
| Exp-5cls | 전체 5종 (현재) | 완전 분리수거 시스템 |

**각 실험에서 측정할 항목:**

```python
# 임계값별 Precision-Recall 곡선 분석
thresholds = [0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5]

for conf_thresh in thresholds:
    results = model.val(conf=conf_thresh, iou=0.5)
    print(f"conf={conf_thresh:.2f}  "
          f"P={results.box.mp:.4f}  "
          f"R={results.box.mr:.4f}  "
          f"F1={2*P*R/(P+R):.4f}")
```

### 7.3 클래스별 신뢰도 분포 분석

형광등은 다른 클래스와 형태가 뚜렷이 달라 신뢰도가 높은 반면,  
비닐은 형태가 불규칙하여 신뢰도 분포가 넓게 퍼지는 경향이 예상된다.

```
예상 신뢰도 분포 (5클래스 모델 기준)

형광등  ████████████████  (좁고 높음, 평균 ~0.88)
유리병  ██████████████    (중간,     평균 ~0.82)
캔류    █████████████     (중간,     평균 ~0.80)
고철류  ████████          (넓고 낮음, 평균 ~0.72)
비닐    ██████            (매우 넓음, 평균 ~0.65)

→ 클래스별 개별 임계값 적용 검토:
  형광등: conf ≥ 0.50
  유리병: conf ≥ 0.40
  캔류:   conf ≥ 0.40
  고철류: conf ≥ 0.30
  비닐:   conf ≥ 0.25
```

### 7.4 최적 임계값 탐색 코드 예시

```python
from sklearn.metrics import f1_score
import numpy as np

def find_optimal_conf(model, val_loader, class_id, conf_range):
    """특정 클래스에 대한 최적 신뢰도 임계값 탐색"""
    best_f1, best_conf = 0.0, 0.25
    for conf in conf_range:
        preds = model.val(conf=conf, classes=[class_id])
        f1 = preds.box.f1.mean()
        if f1 > best_f1:
            best_f1, best_conf = f1, conf
    return best_conf, best_f1

conf_range = np.arange(0.10, 0.60, 0.05)
for cls_id, cls_name in enumerate(['고철류','비닐','유리병','캔류','형광등']):
    opt_conf, opt_f1 = find_optimal_conf(model, val_loader, cls_id, conf_range)
    print(f"{cls_name}: 최적 conf={opt_conf:.2f}, F1={opt_f1:.4f}")
```

---

## 8. 과제 F — 2단계 탐지 아키텍처 (분리수거통 이상 감지 서비스)

### 8.1 서비스 개요

> **"분리수거통에 올바른 폐기물만 들어 있는가?"**

단순 폐기물 탐지를 넘어, **분리수거통별 허용 클래스를 위반하는 이상 폐기물을 실시간으로 감지**하는 서비스를 구축한다.

### 8.2 전체 시스템 아키텍처

```
입력 (CCTV / 고정 카메라)
        │
        ▼
┌────────────────────────────┐
│  Stage 1: 객체 탐지        │  ← YOLO26s (상시 실행)
│  "어떤 폐기물이 어디에?"   │     30fps, 엣지 디바이스 실행
└────────────┬───────────────┘
             │ 탐지된 클래스 목록
             ▼
┌────────────────────────────┐
│  규칙 엔진: 이상 판단      │  ← 수거통별 허용 클래스 DB
│  허용 클래스 ≠ 탐지 클래스 │     즉각 응답 (1ms 미만)
└────────────┬───────────────┘
             │ 이상 감지 트리거
             ▼
┌────────────────────────────┐
│  Stage 2: 상황 확인        │  ← VLM 또는 고해상도 분석
│  "정말 이상 폐기물인가?"   │     트리거 시에만 실행
└────────────┬───────────────┘
             │
             ▼
        알림 발송 / 리포트 저장
```

### 8.3 수거통별 허용 클래스 정의

```python
# 분리수거 규칙 정의
BIN_RULES = {
    "bin_metal":  {"allowed": ["캔류", "고철류"],  "label": "금속류 전용"},
    "bin_glass":  {"allowed": ["유리병"],           "label": "유리 전용"},
    "bin_vinyl":  {"allowed": ["비닐"],              "label": "비닐 전용"},
    "bin_hazard": {"allowed": ["형광등"],            "label": "형광등 전용"},
}

def check_anomaly(detected_classes, bin_id):
    allowed = BIN_RULES[bin_id]["allowed"]
    anomalies = [cls for cls in detected_classes if cls not in allowed]
    return anomalies  # 빈 리스트면 정상
```

### 8.4 트리거 조건 (Stage 2 호출 기준)

```
트리거 조건 (OR 연산):

 ① 허용 외 클래스 탐지
    → 수거통 내 탐지된 클래스 ∩ 비허용 클래스 ≠ ∅

 ② 고신뢰도 이상 탐지
    → 허용 외 클래스의 confidence ≥ 0.5
    → 즉시 알림 (Stage 2 생략 가능)

 ③ 저신뢰도 이상 탐지 (불확실)
    → 0.25 ≤ confidence < 0.5 인 이상 클래스 존재
    → Stage 2로 재확인 요청

 ④ 장시간 변화 없음
    → 수거통이 N분 이상 가득 찬 상태 유지
    → 수거 요청 알림 (폐기물 종류 무관)
```

### 8.5 Stage 2 모델 선택

| 모델 | 특징 | 추론 속도 | 추천 용도 |
|------|------|----------|----------|
| **YOLO26s (crop + 재분류)** | 의심 영역 잘라서 재탐지 | ~50ms | 가장 간단한 구현 |
| **CLIP (zero-shot)** | 텍스트-이미지 매칭으로 클래스 재확인 | ~100ms | 미정의 이상 탐지 |
| **Qwen2.5-VL (7B)** | 장면 자연어 설명 + 이상 판단 | ~400ms | 리포트 생성 필요 시 |

> 권장: 초기에는 **YOLO26s crop 재분류** 방식으로 시작 (구현 간단, 속도 빠름),  
> 오탐 해결이 필요하면 **CLIP zero-shot** 방식으로 전환

### 8.6 알림 시스템

```
🟢 정상 (NORMAL)
└── 로그 기록 (수거통 상태 정상)

🟡 주의 (CAUTION)
├── 관리자 대시보드 황색 표시
└── 저신뢰도 이상 탐지 → Stage 2 재확인 진행 중

🟠 이상 감지 (ANOMALY)
├── 관리자 앱 푸시 알림
├── 이상 클래스 + 위치 정보 포함
└── 캡처 이미지 첨부 저장

🔴 긴급 (URGENT — 위험 폐기물 혼입)
├─ 형광등이 금속/유리통에 혼입 → 수은 오염 위험
├─ 즉시 경보 + 관리자 전화 알림
└─ 영상 클립 자동 저장 (전후 30초)
```

---

## 9. 학습 파이프라인 (개선 버전)

### 9.1 데이터 전처리 파이프라인 개선

```bash
# Step 1: 다중 출처 데이터 수집 및 클래스 통합
python merge_datasets.py \
    --sources 71385 495 71362 \
    --class-map class_mapping.yaml \
    --output data/merged/

# Step 2: 해상도 필터 + PIL 무결성 검사 (기존 json_to_yolo.py 확장)
python json_to_yolo.py          # 기존 스크립트 (71385용)
python json_to_yolo_495.py      # 신규 스크립트 (495용)
python video_to_frames.py       # 71362 영상 → 프레임 추출

# Step 3: 다중 클래스 합성 이미지 생성
python generate_multi_class.py \
    --single-class-dir data/images/ \
    --output-count 3000 \
    --min-classes 2 --max-classes 5

# Step 4: 세션 단위 train/valid/test 분리 (기존 split_dataset.py)
python split_dataset.py  # 70:15:15

# Step 5: 클래스 수별 모델 학습
for N_CLS in 2 3 4 5; do
    yolo detect train \
        data=data/dataset_${N_CLS}cls.yaml \
        model=yolo26s.pt \
        epochs=100 \
        imgsz=640 \
        batch=32 \
        device=0
done
```

### 9.2 학습 설정 변경 사항

| 파라미터 | 현재 값 | 개선 제안 | 이유 |
|---------|--------|----------|------|
| `epochs` | 50 | 100 | 다중 출처 데이터 학습 안정화 |
| `imgsz` | 640 | 640~1024 | 형광등(가는 형태) 탐지 정밀도 |
| `mosaic` | 1.0 | 1.0 | 다중 클래스 공존 효과 유지 |
| `degrees` | 0.0 | 30.0 | 각도 다양성 증강 |
| `flipud` | 0.0 | 0.1 | 뒤집힌 폐기물 대응 |
| `hsv_v` | 0.4 | 0.6 | 조명 다양성 강화 |
| `optimizer` | SGD | AdamW | 다양한 출처 혼합 학습 안정화 |

### 9.3 평가 체계 (개선)

```python
# 기존 평가
results = model.val()  # 단일 mAP

# 개선된 평가
metrics = {
    'mAP50_overall':   model.val(data='full_test.yaml').box.map50,
    'mAP50_single':    model.val(data='single_cls_test.yaml').box.map50,
    'mAP50_multi':     model.val(data='multi_cls_test.yaml').box.map50,
    'recall_anomaly':  evaluate_anomaly_recall(model, anomaly_test_set),
    'fpr_normal':      evaluate_fpr(model, normal_test_set),
}
```

---

## 10. 하드웨어 및 배포 구성

### 10.1 배포 환경 옵션

| 배포 형태 | 하드웨어 | Stage 1 | Stage 2 | 적합 시나리오 |
|----------|---------|---------|---------|-------------|
| **엣지 단독** | Jetson Orin NX 8GB | YOLO26s (INT8) | CLIP (경량) | 소규모 단지, 저비용 |
| **엣지 + 서버** | Jetson + RTX 4060 서버 | YOLO26s | Qwen2.5-VL | 중규모 단지 |
| **클라우드 기반** | AWS EC2 + GPU | YOLO26m | GPT-4o Vision | 대규모 공공시설 |
| **현재 개발 환경** | WSL2 + RTX 4060 Ti 8GB | YOLO26s | 미구현 | 개발·실험 단계 |

### 10.2 엣지 배포 최적화

```bash
# TensorRT 변환 (Jetson 배포용)
yolo export \
    model=runs/waste_yolo26/yolo26s_ep100/weights/best.pt \
    format=engine \
    device=0 \
    half=True \
    int8=True \
    imgsz=640

# ONNX 변환 (범용)
yolo export \
    model=best.pt \
    format=onnx \
    opset=17 \
    simplify=True
```

### 10.3 실시간 처리 성능 목표

| 환경 | 목표 FPS | 현재 추정 FPS | 비고 |
|------|---------|-------------|------|
| RTX 4060 Ti (개발) | ≥ 60 | ~120 (YOLO26s, fp16) | 충분 |
| Jetson Orin NX (배포) | ≥ 30 | ~40 (INT8 TensorRT) | 목표 달성 가능 |
| CPU only (저가형) | ≥ 10 | ~8 (YOLO26s, ONNX) | YOLO26n으로 교체 검토 |

---

## 11. 성능 목표

### 11.1 모델 성능 목표

| 지표 | 현재 (baseline) | 단기 목표 (과제 A~C 완료) | 장기 목표 (과제 F 완료) |
|------|----------------|--------------------------|------------------------|
| mAP@50 (스튜디오) | **0.9624** | ≥ 0.95 (혼합 후 유지) | ≥ 0.93 |
| mAP@50 (실환경) | 미측정 | ≥ 0.80 | ≥ 0.88 |
| mAP@50 (단일 클래스) | 0.9624 | ≥ 0.95 | ≥ 0.95 |
| mAP@50 (다중 클래스) | 미측정 | ≥ 0.85 | ≥ 0.90 |
| 이상 감지 Recall | 미측정 | — | ≥ 0.95 |
| 이상 감지 FPR | 미측정 | — | ≤ 0.10 |
| 처리 속도 (엣지) | 미측정 | ≥ 30 fps | ≥ 30 fps |

### 11.2 데이터 품질 목표

| 항목 | 현재 | 목표 |
|------|------|------|
| 총 이미지 수 | 10,000장 | 25,000장 이상 |
| 출처 수 | 1개 | 3개 (71385 + 495 + 71362) |
| 다중 클래스 공존 비율 | ~5% (mosaic 효과) | 40% 이상 |
| 저조도 이미지 비율 | ~2% | 15% 이상 |
| 비정립 각도 비율 | ~10% | 30% 이상 |

---

## 12. 개발 로드맵

```
Phase 1 (2주)  데이터셋 확장 & 재학습
├── [ ] AI Hub dataSetSn=495, 71362 EDA 수행
├── [ ] 클래스 매핑 테이블 확정 (class_mapping.yaml)
├── [ ] 다중 출처 json_to_yolo 변환 스크립트 작성
├── [ ] 혼합 데이터셋으로 YOLO26s 재학습 (100 epochs)
└── [ ] 스튜디오 vs 실환경 mAP 비교 분석

Phase 2 (2주)  다중 클래스 공존 & 증강
├── [ ] Copy-Paste 합성 스크립트 작성
├── [ ] 단일/다중 클래스 테스트셋 분리 구성
├── [ ] 조명·각도 증강 파라미터 최적화 (그리드 서치)
└── [ ] 단일 vs 다중 클래스 mAP 비교 리포트

Phase 3 (2주)  문제 정의 확정 & 임계값 최적화
├── [ ] 배포 환경 인터뷰 (관리자/운영자 요구사항 수집)
├── [ ] 클래스 수별 (2/3/4/5cls) 모델 학습 및 비교
├── [ ] 클래스별 최적 신뢰도 임계값 탐색
├── [ ] 이상 감지 규칙 엔진 설계
└── [ ] Recall vs Precision 트레이드오프 분석

Phase 4 (4주)  2단계 아키텍처 구현
├── [ ] Stage 1: YOLO26s + 규칙 엔진 연동
├── [ ] Stage 2: CLIP 또는 YOLO26s crop 재분류 구현
├── [ ] 알림 시스템 프로토타입 (Slack/앱 푸시)
├── [ ] 관제 대시보드 MVP 개발
└── [ ] 파일럿 테스트 (실제 분리수거 환경)
```

### 우선순위 요약

| 우선순위 | 과제 | 기대 효과 | 난이도 |
|---------|------|----------|--------|
| ⭐⭐⭐ 최고 | 과제 D: 문제 정의 확정 | 모든 후속 작업 방향 결정 | 낮음 |
| ⭐⭐⭐ 최고 | 과제 E: 임계값 최적화 | 즉시 실환경 성능 향상 가능 | 낮음 |
| ⭐⭐ 높음 | 과제 B: 다중 클래스 공존 | 실환경 일반화 핵심 | 중간 |
| ⭐⭐ 높음 | 과제 A: 데이터셋 확장 | 데이터 다양성 근본 해결 | 중간 |
| ⭐ 중간 | 과제 C: 조명·각도 증강 | 코드 수정으로 즉시 적용 가능 | 낮음 |
| ⭐ 중간 | 과제 F: 2단계 아키텍처 | 서비스 완성도 최고 | 높음 |